In [ ]:
import os
import json
from typing import List, Dict, Any, Optional

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate

from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector

In [ ]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

In [ ]:
NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

graph

In [ ]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm(model_path="./hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

In [ ]:
vector_store = Neo4jVector.from_existing_graph(
    embedding=embedding,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="chunk_vector_index",
    node_label="Chunk",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

# search chunks by similarity
base_vector_retriever = vector_store.as_retriever(search_kwargs={"k": 6})

"vector retriever ready"


In [ ]:
def format_docs(docs: List[Document]) -> str:
    parts = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        cid = d.metadata.get("chunk_id", "")
        parts.append(f"[{i}] source={src} chunk_id={cid}\n{d.page_content}".strip())
    return "\n\n---\n\n".join(parts)

Graph expansion Cypher

In [ ]:
ALLOWED_REL_TYPES = [
    "HAS_SYMPTOM",
    "TRIGGERS",
    "AFFECTS",
    "CAUSED_BY",
    "RESOLVED_BY",
    "REQUIRES_TOOL",
    "USES_MATERIAL",
    "HAS_WARNING",
    "RELATED_TO",
    "NOT_A_DEFECT",
]

def fetch_graph_evidence_for_chunks(
    chunk_ids: List[str],
    hop: int = 1,
    per_chunk_entities: int = 8,
    per_entity_rels: int = 12,
) -> List[Document]:
    """
    Extend evidence for the hit chunks from the Neo4j graph and return a list of Documents (to be merged with vector chunk docs). 
    The output is Documents, ensuring they can be directly consumed by format_docs.
    """
    if not chunk_ids:
        return []

    # 1) Chunk -> Entities (MENTIONS)
    q_entities = """
    UNWIND $chunk_ids AS cid
    MATCH (c:Chunk {chunk_id: cid})-[m:MENTIONS]->(e:Entity)
    WITH cid, c, e, m
    ORDER BY cid, e.type, e.name
    RETURN cid AS chunk_id,
           c.source AS source,
           collect({
             uid: e.uid, type: e.type, name: e.name,
             evidence: coalesce(m.evidence,'')
           })[0..$per_chunk_entities] AS entities
    """

    entity_rows = graph.query(q_entities, params={
        "chunk_ids": chunk_ids,
        "per_chunk_entities": per_chunk_entities
    })

    # list all entities
    all_entity_uids = []
    chunk_to_entities = {}
    for row in entity_rows:
        ents = row.get("entities", []) or []
        chunk_to_entities[row["chunk_id"]] = ents
        for e in ents:
            if e.get("uid"):
                all_entity_uids.append(e["uid"])

    all_entity_uids = list(dict.fromkeys(all_entity_uids))  # unique, keep order

    # 2) Entities -> (Entity)-[REL]->(Entity) 1 hop
    q_rels_1hop = f"""
    UNWIND $entity_uids AS uid
    MATCH (a:Entity {{uid: uid}})-[r]->(b:Entity)
    WHERE type(r) IN $allowed_rels
    RETURN a.uid AS source_uid, a.name AS source_name, a.type AS source_type,
           type(r) AS rel_type, coalesce(r.evidence,'') AS evidence,
           b.uid AS target_uid, b.name AS target_name, b.type AS target_type
    LIMIT $limit
    """

    rel_rows = graph.query(q_rels_1hop, params={
        "entity_uids": all_entity_uids,
        "allowed_rels": ALLOWED_REL_TYPES,
        "limit": max(50, len(all_entity_uids) * per_entity_rels)
    })

    # get contexual content from Graph
    chunk_graph_docs: List[Document] = []

    
    uid_to_rels = {}
    for rr in rel_rows:
        uid_to_rels.setdefault(rr["source_uid"], []).append(rr)

    for cid in chunk_ids:
        ents = chunk_to_entities.get(cid, [])
        if not ents:
            continue

        lines = []
        lines.append(f"Graph evidence for chunk_id={cid}")
        lines.append("Mentioned entities:")
        for e in ents:
            ev = e.get("evidence", "")
            ev_txt = f' (evidence: "{ev}")' if ev else ""
            lines.append(f"- {e['type']}: {e['name']}{ev_txt}")

        lines.append("")
        lines.append("Expanded relations (1-hop):")
        used_any = False

        for e in ents:
            uid = e.get("uid")
            if not uid:
                continue
            for r in (uid_to_rels.get(uid, []) or [])[:per_entity_rels]:
                used_any = True
                ev = r.get("evidence", "")
                ev_txt = f' (evidence: "{ev}")' if ev else ""
                lines.append(
                    f"- ({r['source_type']}:{r['source_name']}) -[{r['rel_type']}]-> ({r['target_type']}:{r['target_name']}){ev_txt}"
                )

        if not used_any:
            lines.append("- (no relations found)")

        # Graph doc metadata
        chunk_graph_docs.append(
            Document(
                page_content="\n".join(lines).strip(),
                metadata={"chunk_id": cid, "source": (ents[0].get("source") if False else "graph")}
            )
        )

    return chunk_graph_docs


In [ ]:
def graph_retrieve(question: str, k: int = 6, hop: int = 1) -> List[Document]:
    # 1) vector retrieve chunks
    chunk_docs = base_vector_retriever.get_relevant_documents(question)

    # 2) collect chunk_ids
    chunk_ids = []
    for d in chunk_docs:
        cid = d.metadata.get("chunk_id") or d.metadata.get("chunkId") or d.metadata.get("id")
        if cid:
            chunk_ids.append(cid)

   
    chunk_ids = list(dict.fromkeys(chunk_ids))

    # 3) graph expansion evidence docs
    graph_docs = fetch_graph_evidence_for_chunks(
        chunk_ids=chunk_ids,
        hop=hop,
        per_chunk_entities=8,
        per_entity_rels=10,
    )

    # 4) merge: chunk docs first, then graph docs
    return chunk_docs + graph_docs

retriever = RunnableLambda(lambda q: graph_retrieve(q, k=6, hop=1))

print("graph-based retriever runnable ready")


Build Chain

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant specializing in 3D printing operations.
Use the following pieces of retrieved context to answer the question.
If you don’t know the answer, just say that you don’t know.
Use two sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

rag_chain


In [ ]:
rag_chain.invoke("What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?")

In [ ]:
import pandas as pd
import time
from ragas import EvaluationDataset
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model


def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),
         FactualCorrectness(mode = 'f1', atomicity='high', coverage='high')]

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, rs_times = get_eval_ds(data_path, retriever, rag_chain)

In [ ]:
result = get_eval_result(eval_ds, metrics)
result

In [ ]:
df = result.to_pandas()
df['response_time'] = rs_times
df.to_csv("./evaluate_results/09_KG_test/llm_based_test.csv")